In [ ]:
import numpy as np
import h5py
import time
import pickle

import sys
from pathlib import Path

SRC = Path.cwd().parent
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from matplotlib import pyplot as plt
from helpers.simulation import *
from helpers.qtransform import *
from dataset_generation.gw_class_gen import *
from config import *

#from tensorflow.keras.applications import InceptionV3
#from tensorflow.keras.models import Model

import torchvision.models as models

import warnings
warnings.filterwarnings("ignore")

In [3]:
lm_path = DIST / 'model_weights' / 'lm_classifier.hdf5'

lm_model = h5py.File(lm_path, "r")

In [5]:
model = InceptionV3(weights=None, include_top=True, input_shape=(256, 256, 1), classes=9)
model.load_weights(lm_path)

In [12]:
old_base = Model(
    inputs=model.input,
    outputs=model.get_layer("mixed10").output
)

new_base = InceptionV3(
    weights=None,
    include_top=True,
    input_shape=(256,256,3)
)

In [14]:
for layer_new in new_base.layers:
    if layer_new.name == "conv2d":
        continue   # skip first layer

    try:
        layer_old = old_base.get_layer(layer_new.name)
        if layer_new.get_weights() and layer_old.get_weights():
            layer_new.set_weights(layer_old.get_weights())
    except ValueError:
        pass

In [15]:
for layer in new_base.layers[:10]:
    print(layer.name, len(layer.get_weights()))

input_layer_2 0
conv2d_188 1
batch_normalization_188 3
activation_188 0
conv2d_189 1
batch_normalization_189 3
activation_189 0
conv2d_190 1
batch_normalization_190 3
activation_190 0
